# 🔎 Week 4B: Verification and validation of Agentic GeoAI
How do we know an agent-generated geospatial result is credible?

This notebook follows the Week 4 overview of Agentic GeoAI.

Last week, our agent could **produce** an analysis. This week, the emphasis changes:

> **Do not ask only whether the agent completed the task. Ask whether the result is correct, plausible, robust, and defensible.**

### 🎯 Learning objectives

By the end of this exercise, you should be able to:

1. distinguish **verification** from **validation**;
2. identify common failure modes in agent-generated geospatial analysis;
3. design checks before an agent begins its analysis;
4. evaluate geometry, distributions, topology, units, CRS, and spatial plausibility;
5. use **secondary/independent datasets** as evidence;
6. test sensitivity to modeling assumptions;
7. distinguish internal consistency from external validity;
8. use an agent as both an analyst and a critic without treating self-review as independent validation.

## 📂 Before we begin

Open this notebook from the course repository using your course Python environment. The small examples use pandas, GeoPandas, and Shapely; they need no network access and do not modify your routing files. Run code cells in order. Complete the **Your response** cells yourself before comparing with an agent.

We will distinguish verification from validation, practice with synthetic data, and audit the saved Week 3 outputs. See [04_01](04_01_agentic_geoai_applications.ipynb) for the application framework and [03_02](03_02_ggs662_agentic_application_route_planning.ipynb) for the routing exercise.

## 🧭 1. Verification versus validation

We will use a practical distinction:

### Verification
**Did we perform the analysis correctly?**

Examples:
- Did the code run as intended?
- Are distances measured in meters rather than degrees?
- Do route endpoints match the requested locations?
- Are geometries valid?
- Did a spatial join use the correct predicate?
- Did the route remain inside the intended search area?

### Validation
**Does the analysis adequately represent the real-world problem?**

Examples:
- Are mapped roads actually present?
- Does the population surface resemble other credible population datasets?
- Is the selected route physically plausible?
- Are the assumptions about co-location reasonable?
- Does an independently derived dataset support the same conclusion?

A workflow can be **verified but invalid**.

> Correct code can still answer the wrong question, use poor evidence, or produce an unrealistic model of the world.

**Research connection:** [Oreskes, Shrader-Frechette, and Belitz (1994)](https://doi.org/10.1126/science.263.5147.641) challenge the idea that agreement with observations can conclusively validate a model of an open natural system. Our classroom distinction is operational: check implementation against a specification, then assess fitness for a stated use. Passing checks supports bounded claims; it does not prove the model universally true.

### ✍️ Exercise: verification or validation?

Classify these checks and explain your reasoning: (1) recalculate cost from length; (2) inspect independent imagery for a mapped road; (3) compare route endpoints with the specification; (4) assess whether the data resolution supports a construction decision. Can one check contribute to both categories?

**Your response**

| Check | Verification / validation | Reason |
|---|---|---|
| 1 | … | … |
| 2 | … | … |
| 3 | … | … |
| 4 | … | … |

## 🌍 2. Validation should not begin at the end

A weak workflow is:

```text
TASK → AGENT → RESULT → "Does this look okay?"
```

A stronger workflow is:

```text
              EXPECTATIONS
                  │
                  ▼
DATA ──→ AGENT WORKFLOW ──→ RESULT
 │             │                │
 │             │                │
 ▼             ▼                ▼
DATA QA     PROCESS QA      OUTPUT QA
 │             │                │
 └──────→ INDEPENDENT / SECONDARY EVIDENCE
                          │
                          ▼
                 SENSITIVITY / ROBUSTNESS
                          │
                          ▼
                     HUMAN REVIEW
```

Before analysis starts, define what you expect to observe and what evidence could reveal that the result is wrong.

## 🔬 3. A practical validation stack for Agentic GeoAI

We will organize validation into seven layers.

| Layer | Core question | Examples |
|---|---|---|
| 1. Provenance | Where did the evidence come from? | source, date, coverage, license |
| 2. Data QA | Are the inputs plausible? | distributions, missingness, geometry |
| 3. Process verification | Did the workflow do what it claimed? | CRS, units, parameters, code |
| 4. Spatial sanity | Does the geography make sense? | topology, extent, distances, patterns |
| 5. External validation | Does independent evidence agree? | second dataset, imagery, authoritative source |
| 6. Sensitivity | Does the conclusion survive reasonable changes? | weights, buffers, thresholds |
| 7. Claim audit | Is the conclusion supported by the evidence? | uncertainty, access rights, causality |

A credible agentic workflow should leave evidence at each layer.

## 🗺️ 4. Start with expected distributions and spatial structure

Before asking an agent to analyze a dataset, ask:

> **What should this dataset look like if it is approximately correct?**

Examples:

### Population
We normally expect:
- non-negative values;
- many low-density locations and fewer very high-density locations;
- clusters around settlements;
- major urban areas to be visible;
- spatial continuity rather than random noise.

### Roads
We expect:
- connected linear networks;
- greater density around settlements;
- road classes to have plausible relative frequencies;
- very few isolated microscopic fragments if the extraction is sensible.

### Elevation
We expect:
- locally smooth spatial variation;
- plausible regional ranges;
- no abrupt rectangular discontinuities caused by tiling/data errors.

### Fiber route
We expect:
- a continuous line between the requested endpoints;
- no teleportation/jumps;
- no unexplained loops;
- a plausible length relative to straight-line distance;
- plausible interaction with roads, railways, terrain, settlements, and other constraints.

These expectations guide **checks**, not automatic rejection rules. Sparse roads, sharp elevation changes, and unusual population values may be real. Investigate an unexpected pattern using its location, scale, units, and provenance before labeling it an error.

## 🤖 5. Inspect the data before modeling

For every important layer, the agent should report basic diagnostics.

### Tabular diagnostics
- number of features;
- missing values;
- duplicates;
- minimum / maximum;
- mean / median;
- quantiles;
- categorical frequencies.

### Spatial diagnostics
- CRS;
- geographic extent;
- geometry types;
- invalid/empty geometries;
- spatial distribution;
- obvious outliers;
- unexpected gaps;
- overlap with the study area.

### Why distributions matter

Suppose an agent downloads a population dataset with values:

`0, 0, 0, 1, 2, 3, 5, 8, 950000`

The code may work perfectly, but the extreme value should trigger investigation before it influences an analysis. Population is a diagnostic example here; it is not a cost or benefit input to our Week 3 routing model.

In [ ]:
# Generic numeric diagnostics the students/agent can adapt

def describe_numeric(series):
    return series.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )

# A small teaching example; no downloads are needed.
import pandas as pd

population = pd.Series([0, 0, 0, 1, 2, 3, 5, 8, 950000], name="population")
print(describe_numeric(population))

### ✍️ Exercise: investigate an outlier

Compare the mean, median, and maximum. List two plausible explanations for 950000 and one piece of metadata that would help distinguish them. Would you delete the value immediately? Explain.

**Your response**

* Observed pattern: …
* Two explanations: …
* Metadata needed: …
* Decision: …

## 🧰 6. Spatial sanity checks

Geospatial workflows introduce failure modes that ordinary tabular analysis does not.

### CRS and units
Ask:
- What CRS is each dataset in?
- Are distance/area calculations being performed in an appropriate projected CRS?
- Is a 1,000-unit buffer actually 1,000 meters?

### Geometry
Ask:
- Are geometries valid?
- Are there empty geometries?
- Are LineStrings unexpectedly fragmented?
- Are polygons self-intersecting?

### Extent
Ask:
- Do all datasets overlap the intended study area?
- Has longitude/latitude been reversed?
- Is a layer accidentally in the ocean or another continent?

### Topology and connectivity
Ask:
- Does a route form one connected path?
- Are road-network components disconnected?
- Are there jumps between infrastructure segments?

### Scale
Ask:
- Is the resolution appropriate for the question?
- Are we drawing engineering conclusions from data that are too coarse?

In [ ]:
# Example GeoDataFrame checks

def spatial_qa(gdf, name="layer"):
    print(f"--- {name} ---")
    print("Features:", len(gdf))
    print("CRS:", gdf.crs)
    print("Geometry types:")
    print(gdf.geom_type.value_counts(dropna=False))
    print("Empty geometries:", int(gdf.geometry.is_empty.sum()))
    print("Missing geometries:", int(gdf.geometry.isna().sum()))
    usable = gdf.geometry.notna() & ~gdf.geometry.is_empty
    print("Invalid nonempty geometries:", int((usable & ~gdf.geometry.is_valid).sum()))
    print("Bounds:", gdf.total_bounds)

# Synthetic projected coordinates in metres; this is not a real road dataset.
import geopandas as gpd
from shapely.geometry import LineString

demo_lines = gpd.GeoDataFrame(
    {"name": ["segment_a", "segment_b", "missing"]},
    geometry=[LineString([(500000, 9500000), (500100, 9500000)]),
              LineString([(500120, 9500000), (500220, 9500000)]), None],
    crs="EPSG:32737",
)
spatial_qa(demo_lines, "Synthetic corridor")

### ✍️ Exercise: valid geometry, disconnected network

The two synthetic lines are individually valid. Sketch their coordinates and calculate the gap by hand. Would that gap qualify under 50 m and 10 m tolerances, assuming study-area containment? Why does geometry validity alone miss this problem?

**Your response**

* Hand-calculated gap: … m
* At 50 m: …
* At 10 m: …
* Additional topology check: …

### 🧮 A known case: count a connector once

Before running the next cell, calculate the total length and classroom cost of two 100 m segments joined by a 20 m gap. This is an arithmetic oracle, not a test of a shortest-path implementation. The actual routing code still needs the synthetic cases specified in `AGENTS.md`.

In [ ]:
# Expected values were calculated by hand: 220 m and USD 2,200.
from math import isclose

source_lengths_m = [100.0, 100.0]
gap_length_m = 20.0
rate_usd_per_m = 10.0
total_length_m = sum(source_lengths_m) + gap_length_m
estimated_cost_usd = total_length_m * rate_usd_per_m
assert isclose(total_length_m, 220.0, abs_tol=0.01)
assert isclose(estimated_cost_usd, 2200.0, abs_tol=0.01)
print(f"Length: {total_length_m:g} m; cost: USD {estimated_cost_usd:,.0f}")

# Deliberately faulty report: the gap was included twice.
reported_length_m = sum(source_lengths_m) + 2 * gap_length_m
print("Faulty report passes:", isclose(reported_length_m, 220.0, abs_tol=0.01))

### ✍️ Exercise: have a go at a second known case

In the cell below, use source segments of 80 m and 120 m, a 30 m gap, and USD 10/m. Write down your expected values first, then add assertions. Explain which error these checks would catch and which real-world claim they cannot establish.

In [ ]:
# Enter your attempt here.
# Expected total length: ...
# Expected cost: ...

## 🛰️ 7. Independent and secondary datasets

One of the most important principles for this class is:

> **Do not validate a dataset solely against itself or against another output derived from the same source.**

Examples:

| Primary evidence | Possible secondary evidence |
|---|---|
| OSM settlements | GHSL / WorldPop / census |
| OSM roads | satellite imagery / authoritative roads |
| OSM railway | imagery / operator or government data |
| OSM power lines | imagery / utility or energy datasets |
| Sentinel-2 classification | Landsat / high-resolution imagery / reference labels |
| DEM-derived slope | alternative DEM / contours / known elevation points |
| Agent-generated route | existing feasibility route / imagery / engineering evidence |

Agreement does not prove correctness, but disagreement tells us where to investigate.

**GIScience reading:** [Haklay (2010)](https://doi.org/10.1068/b35097) compares OpenStreetMap with Ordnance Survey data. Read it as an example of designing a spatial data-quality comparison, not as evidence that today's OSM data in our Kenyan study area have the same quality. Ask which reference source, spatial sample, and quality measure would be appropriate here.

## 🔎 8. Independence matters

Not every “second dataset” provides independent validation.

For example:

```text
OpenStreetMap
     │
     ├── Dataset A
     │
     └── Dataset B derived from OSM
```

Comparing A and B may test processing consistency, but it does **not** provide strong external validation.

Stronger:

```text
OpenStreetMap roads ──────┐
                          ├── compare
Independent imagery ──────┘
```

Ask the agent:

> **Is the validation evidence genuinely independent of the evidence used to construct the result?**

**Extension to predictive GeoAI:** [Roberts et al. (2017)](https://doi.org/10.1111/ecog.02881) discuss cross-validation for structured data. Nearby training and test observations may share spatial structure, so a random split can give an overly optimistic estimate of transfer to new areas. This is a different independence issue from shared source provenance. Spatial blocking can address dependence between samples; it does not turn two derivatives of the same dataset into independent sources. This reading extends the lesson to predictive models; our routing exercise does not train a model.

### ✍️ Exercise: is the evidence independent?

Compare (a) two road exports from the same OSM snapshot, (b) OSM roads and independently acquired imagery, and (c) two products derived from the same satellite image. What can each comparison establish? Record shared sources, date mismatches, and uncertainty.

**Your response**

| Pair | Shared evidence | Useful check | Limitation |
|---|---|---|---|
| a | … | … | … |
| b | … | … | … |
| c | … | … | … |

## ⚠️ 9. Quantitative comparison

Visual inspection is useful but insufficient.

Depending on the problem, quantitative validation could include:

- positional error;
- completeness;
- precision / recall;
- intersection-over-union;
- correlation;
- RMSE / MAE;
- confusion matrices;
- network connectivity;
- route overlap;
- percentage of route near a mapped corridor;
- difference in total length;
- distributional comparison.

The metric must match the scientific question.

A high correlation, for example, does not demonstrate positional accuracy.

**Remote-sensing reading:** [Olofsson et al. (2014)](https://doi.org/10.1016/j.rse.2014.02.015) connect land-change accuracy assessment to sampling design, reference labels, error matrices, and uncertainty in area estimates. A confusion matrix is useful only when its sample and reference evidence support the intended inference. These classification methods are an extension for EO applications, not a replacement for route connectivity and length checks.

## 💡 10. Sensitivity analysis: change one assumption

Return to the **simple Week 3 model**: separate road, power, and rail routes, plus the shortened baseline. All use one positive cost per metre. Settlements provide map context only.

For a sensitivity rerun, change only `gap_tolerance_m` from its saved value (default 50 m) to 25 m. Keep the exact endpoints, study rectangle, endpoint connector limit (default 500 m), input data, and common rate fixed. If your saved gap is already 25 m, choose another smaller positive value and record it.

Save the rerun under `outputs/week3_routing/sensitivity_gap_25m/` (or a matching name for your chosen value). Preserve the first comparison. Report changes in status, length, connector use, and the cheapest available candidate. A newly unavailable candidate is a result, not a reason to increase the tolerance.

With a positive uniform rate, the shortest available route is also cheapest. Changing that common rate scales costs but cannot change their ranking when lengths and availability remain fixed. This is different from testing network connectivity.

**Scope of this exercise:** [Saltelli et al. (2019)](https://doi.org/10.1016/j.envsoft.2019.01.012) explain limitations of sensitivity analyses that explore too little of the input space. Our one-setting rerun is a bounded classroom demonstration. It cannot reveal all parameter interactions or establish global robustness. Keep the prescribed exercise unchanged; in a larger study, justify parameter ranges and a design that examines joint effects.

In [ ]:
# Teaching values only: these are not measured Week 3 route lengths.
demo_lengths_m = {"baseline": 1200.0, "roads": 1500.0, "power": 1400.0}
for rate in (5.0, 10.0, 20.0):
    costs = {name: length * rate for name, length in demo_lengths_m.items()}
    print(f"USD {rate:g}/m: {costs}; cheapest = {min(costs, key=costs.get)}")

These rates are invented classroom values, not construction estimates.

## 📊 11. Validate the *claim*, not just the map

Suppose an agent reports:

> “The power-led route is preferable because it follows transmission infrastructure and therefore has existing wayleave rights.”

There are two separate claims:

1. **Spatial claim:** the route is close to mapped transmission infrastructure.
2. **Institutional/legal claim:** access rights exist.

The first may be testable using spatial data.

The second is **not established by proximity**.

A strong agent should therefore report:

> “The route has substantial spatial co-location with mapped transmission infrastructure, which may represent a wayleave opportunity. Actual access rights require independent legal/commercial verification.”

This is part of validation: checking whether the **strength of the conclusion matches the strength of the evidence**.

### ✍️ Exercise: rewrite an unsupported claim

An agent says: “This is the cheapest feasible fiber route because it follows power lines.” Rewrite this in two sentences using only what our uniform-rate model and mapped geometry support. Name the missing evidence needed to establish feasibility.

**Your response**

* Revised claim: …
* Missing evidence: …

## 🎯 12. Reopen the shortened corridor analysis

We are auditing the **exact endpoints of `route1_short.gpkg`**, with the easterly endpoint as the start. The western endpoint is the cut point from 03_01, not Taveta.

Use the five saved 03_01 inputs: `route1_short.gpkg`, `roads.gpkg`, `power_lines.gpkg`, `rail_lines.gpkg`, and `settlements.gpkg`. Read the Week 3 instructions in `AGENTS.md` and the original settings. Recreate the rectangular study area using the saved margin (default 3,000 m), and measure in EPSG:32737.

Inspect `outputs/week3_routing/`: `routes.gpkg`, `route_comparison.csv`, `route_map.png`, `routing_report.md`, and `connectors.gpkg` if connectors were used. The comparison should contain `baseline`, `roads`, `power`, and `rail`. If the previous run used another formulation, document that mismatch before interpreting its results.

If an input is missing, identify its save step in 03_01. If routing outputs are missing, return to 03_02. Do not download substitutes or fabricate routes. You can still complete the synthetic examples and audit design in this notebook.

Our question is: **Which conclusions are supported by the saved evidence, and which require further investigation?**

## 💬 13. Write expectations before reviewing results

Write at least five checks before asking an agent to audit the saved routes. Include an expected outcome, tolerance where appropriate, and the evidence you will inspect.

Consider exact endpoints, continuity, geometry validity, containment, metre-based length, connector limits, cost arithmetic, and the treatment of unavailable routes. Route length should not be shorter than endpoint straight-line distance, but a network route can be longer than the supplied baseline.

For real-world interpretation, specify one claim that geometry alone cannot establish. A 2D crossing may hide a bridge or tunnel; mapped infrastructure does not establish access rights or construction feasibility.

**Your checks, written before the audit**

| Check | Expected result / tolerance | Evidence | What failure would mean |
|---|---|---|---|
| 1 | … | … | … |
| 2 | … | … | … |
| 3 | … | … | … |
| 4 | … | … | … |
| 5 | … | … | … |

* Claim geometry cannot establish: …

## 📝 14. Give the agent an explicit audit task

First complete your own expected checks. Then adapt this instruction with your actual saved paths and settings:

> Read the Week 3 routing instructions in `AGENTS.md` and audit the existing shortened-corridor results. Preserve all source data and the first routing comparison. Save an audit script, its run command and package versions, `validation_table.csv`, `validation_report.md`, and a map under `outputs/week4_validation/`.
>
> Inspect provenance, feature counts, missing/empty/invalid geometries, exclusions, CRS, extent, and component counts before and after gap repairs. Use EPSG:32737, the original study rectangle, and the exact baseline endpoints with the easterly endpoint first. Report missing inputs and their 03_01 save steps; do not download replacements.
>
> Independently recompute geometry lengths and costs. Verify endpoint agreement within 0.01 m, continuity, nonempty valid geometry, containment, and reconstructed length versus traversed-edge sum within 0.01 m. Check every used gap and endpoint connector against its separate saved limit, containment, and counting exactly once. Check baseline connector values are zero. Missing edge evidence is `NOT_CHECKED`, not a pass.
>
> Check all four comparison rows, `ok` statuses for successful routes, and blank lengths/costs for unavailable candidates. Audit the five required synthetic routing cases from AGENTS.md: already connected, gap below the threshold, gap above it, endpoint on a line interior, and disconnected endpoints. Record expected and observed outcomes, including connector length counted once.
>
> Rerun with only the smaller gap tolerance specified above and save separately under `outputs/week3_routing/`. Compare availability, length, connectors, and ranking against the original run. Never alter a threshold to force success.
>
> Identify two possible external evidence sources and explain their provenance, date, coverage, and independence. Compare one specific location using instructor-provided or approved evidence if available. Do not acquire new data automatically; if evidence is absent, state what remains unresolved. Independent evidence is for validation, not a new routing input.
>
> Document method, expected value, observed value, evidence path, and `PASS`, `FAIL`, or `NOT_CHECKED` for each check. Distinguish verified computations, externally supported interpretations, sensitivity, and unresolved assumptions. Show inferred connectors distinctly and retain OpenStreetMap attribution on maps. Do not infer access rights or feasibility from mapped proximity.

Use one focused audit request and, if needed, one repair request describing the observed failure. Inspect the actual files and evidence yourself.

## 📋 15. Make the agent show its evidence

The audit table should let someone else repeat a check. “Looks correct” is not evidence.

| Check | Expected result | Method/evidence | Observed result | Status |
|---|---|---|---|---|
| Endpoints | Agreement within 0.01 m | Exact shortened baseline endpoints and route coordinates | … | NOT_CHECKED |
| Length | Geometry and edge sum agree within 0.01 m | Route geometry, traversed edges | … | NOT_CHECKED |
| Connectors | Within separate limits; counted once | Used connector layer and route edges | … | NOT_CHECKED |
| Costs | Length × common positive rate | Comparison CSV and settings | … | NOT_CHECKED |
| Unavailable routes | Blank lengths/costs | All four CSV rows | … | NOT_CHECKED |
| External evidence | Stated location/date supports a specific claim | Independent source and provenance | … | NOT_CHECKED |
| Gap sensitivity | Only gap setting changed | Original and separate rerun | … | NOT_CHECKED |

Use `FAIL` when evidence contradicts the expected outcome. Use `NOT_CHECKED` when evidence is missing or the check has not been performed. Keep these test statuses separate from the interpretation labels below.

**Agent-evaluation reading:** [AgentBoard (Ma et al., 2024)](https://arxiv.org/abs/2401.13178) evaluates intermediate progress as well as final success. This motivates examining where a workflow failed, rather than recording only whether a file was produced. [AgentBench (Liu et al., 2024)](https://arxiv.org/abs/2308.03688) evaluates agents in multiple interactive environments. Neither benchmark alone establishes the scientific validity of a GIS analysis; task-specific spatial checks remain necessary.

## ✅ 16. A useful classification for the final result

At the end of the audit, classify each important conclusion as:

### VERIFIED
The computational/spatial operation has been checked.

### EXTERNALLY SUPPORTED
Independent evidence supports the underlying real-world interpretation.

### SENSITIVE
The conclusion changes materially under reasonable assumptions.

### UNRESOLVED
Available evidence cannot establish the claim.

These labels describe findings and can overlap: a computation may be VERIFIED while its ranking is SENSITIVE and its real-world feasibility remains UNRESOLVED. Link each label to evidence and scope.

## 🤔 17. Can the same agent validate itself?

An agent can perform useful **self-checks**:

- inspect code;
- rerun calculations;
- detect invalid geometry;
- compare outputs;
- test alternative parameters;
- search for contradictory evidence.

But self-review is not automatically independent validation.

```text
ANALYST AGENT
     │
     ▼
RESULT
     │
     ▼
SAME AGENT REVIEWS RESULT
```

This can catch mistakes, but the same model may reproduce the same assumptions.

A stronger architecture can separate roles:

```text
ANALYST AGENT ──→ RESULT ──→ REVIEWER / VALIDATOR
                                  │
                                  ▼
                           INDEPENDENT DATA
                                  │
                                  ▼
                             HUMAN REVIEW
```

For a first exercise, we do **not** need a complex multi-agent system. The conceptual separation is what matters.

**Research connection:** [Reflexion (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366) uses feedback and stored verbal reflections to improve later attempts. That is a mechanism for revision, not proof of independent validation. [Kao et al. (2026)](https://aclanthology.org/2026.findings-acl.124/) provide an EO-specific evaluation of agent performance. Use these readings to distinguish better task performance from evidence supporting a real-world claim.

## 🛠️ 18. Optional extension: make your audit reusable

Have a go at turning your checks into a reusable Markdown checklist. Save it with the audit deliverables as `validation_checklist.md`.

Specify when each check applies, its required inputs, the evidence it produces, and what happens if the evidence is missing. A reusable instruction should not silently mark an unavailable check as passed.

This can later inform an agent skill. For this class, a plain checklist is enough; no editor configuration or additional agents are needed.

## 📄 19. A starting checklist

```markdown
# Geospatial validation checklist

1. Record the objective, claim boundary, source paths, dates, and settings.
2. Inspect feature counts, missingness, units, CRS, geometry, and extent.
3. Compare observed patterns with expectations written before analysis.
4. Independently check endpoints, topology, lengths, connectors, and costs.
5. Record expected versus observed results with reproducible evidence.
6. Distinguish independent validation from same-source consistency checks.
7. Change one requested setting and preserve the original result.
8. Record PASS, FAIL, or NOT_CHECKED for each applicable check.
9. Separate verified, externally supported, sensitive, and unresolved claims.
10. Preserve inputs and save derived audit outputs separately.
```

Adapt this to a different spatial task. Which checks would change for a raster classification?

## 📦 20. Your class deliverable

Each group should submit:

1. **One audit map** with the original routes, used connectors shown distinctly, available validation evidence, and OpenStreetMap attribution.
2. **One validation table** with expected/observed results, evidence paths, and explicit statuses.
3. **One gap-tolerance sensitivity comparison**, saved separately from the first routing results.
4. **An external-evidence assessment** identifying two candidate sources and their independence. Inspect at least one supplied/approved source if available; otherwise name the missing evidence and leave that claim unresolved.
5. **A reproducible audit script** with its run command and package versions, plus the original and any repair agent instructions.
6. **A short conclusion** (maximum 250 words): what do we trust, what remains uncertain, and what would be needed before an engineering decision?

If Week 3 results are unavailable, submit the completed synthetic work and audit plan, clearly marking the real-route audit and sensitivity comparison incomplete. Synthetic data cannot validate the real corridor.

## 💭 21. Reflection

1. What did the agent originally get right?
2. What looked plausible but failed closer inspection?
3. Which verification check was most useful?
4. Did independent data support the original inputs?
5. Did the preferred route survive sensitivity analysis?
6. Which assumptions remain impossible to validate from open spatial data?
7. Did the agent make any claim stronger than its evidence?
8. Would you allow this system to autonomously make the real routing decision?
9. Which parts should remain human decisions?
10. How would you redesign the agent so validation happens **during** the workflow rather than only afterward?

**Your reflection**

Choose three questions above. Support each answer with a specific finding or unresolved check from your audit.

* Finding and evidence: …
* Limitation and consequence: …
* Change to your next agent instruction: …

## 🏁 22. Take-home message

A successful agent run is not the same thing as a successful analysis.

For Agentic GeoAI, we want:

```text
GOAL
  ↓
PLAN
  ↓
ACT
  ↓
OBSERVE
  ↓
VERIFY ────────┐
  ↓            │
VALIDATE       │
  ↓            │
REVISE ◄───────┘
  ↓
REPORT UNCERTAINTY
```

The aim is not to eliminate human judgment.

The aim is to make the **human–agent workflow scientifically and spatially defensible**.

## 📚 References and suggested reading

Start with Oreskes et al., Haklay, and AgentBoard for the route audit. Olofsson et al. and Roberts et al. extend the discussion to remote sensing and predictive GeoAI.

### Model credibility and spatial data quality

* **Oreskes, N., Shrader-Frechette, K., & Belitz, K. (1994).** [Verification, validation, and confirmation of numerical models in the Earth sciences](https://doi.org/10.1126/science.263.5147.641). *Science*, 263(5147), 641-646. **Peer-reviewed.** Explains why evidence can support a model without proving it universally true; connects to sections 1 and 11.
* **Haklay, M. (2010).** [How good is volunteered geographical information? A comparative study of OpenStreetMap and Ordnance Survey datasets](https://doi.org/10.1068/b35097). *Environment and Planning B: Planning and Design*, 37(4), 682-703. **Peer-reviewed.** A concrete example of spatial data-quality comparison; useful for designing external checks in section 7.
* **Olofsson, P., Foody, G. M., Herold, M., Stehman, S. V., Woodcock, C. E., & Wulder, M. A. (2014).** [Good practices for estimating area and assessing accuracy of land change](https://doi.org/10.1016/j.rse.2014.02.015). *Remote Sensing of Environment*, 148, 42-57. **Peer-reviewed.** Read for sampling, reference data, error matrices, and uncertainty; connects to sections 7-9.
* **Roberts, D. R., et al. (2017).** [Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure](https://doi.org/10.1111/ecog.02881). *Ecography*, 40, 913-929. **Peer-reviewed.** Explains why evaluation splits should account for dependence and the intended prediction task; extends section 8.

### Sensitivity and robustness

* **Saltelli, A., et al. (2019).** [Why so many published sensitivity analyses are false: A systematic review of sensitivity analysis practices](https://doi.org/10.1016/j.envsoft.2019.01.012). *Environmental Modelling & Software*, 114, 29-39. **Peer-reviewed.** Explains the limits of narrow parameter perturbations and why section 10 provides only an introductory check.

### Agent evaluation and feedback

* **Liu, X., et al. (2024).** [AgentBench: Evaluating LLMs as Agents](https://arxiv.org/abs/2308.03688). *ICLR 2024*; arXiv:2308.03688. **Peer-reviewed conference paper; open arXiv version.** Evaluates agents across interactive environments; connects to sections 14-15.
* **Ma, C., et al. (2024).** [AgentBoard: An Analytical Evaluation Board of Multi-turn LLM Agents](https://arxiv.org/abs/2401.13178). *NeurIPS 2024*; arXiv:2401.13178. **Peer-reviewed conference paper; open arXiv version.** Examines intermediate progress and failure points beyond final success; connects to section 15.
* **Shinn, N., Cassano, F., Gopinath, A., Narasimhan, K., & Yao, S. (2023).** [Reflexion: Language Agents with Verbal Reinforcement Learning](https://arxiv.org/abs/2303.11366). *NeurIPS 2023*; arXiv:2303.11366. **Peer-reviewed conference paper; open arXiv version.** Explains feedback and memory for revision; useful for distinguishing self-review from independent validation in section 17.
* **Mohammadi, M., Li, Y., Lo, J., & Yip, W. (2025).** [Evaluation and Benchmarking of LLM Agents: A Survey](https://arxiv.org/abs/2507.21504). arXiv:2507.21504. **Cited in its arXiv version.** Organizes evaluation by objectives and process; useful for extending the audit framework in section 3.

### Agentic GIS and Earth observation

* **Li, Z., & Ning, H. (2023).** [Autonomous GIS: the next-generation AI-powered GIS](https://arxiv.org/abs/2305.06453). arXiv:2305.06453. **Open preprint version.** Provides the autonomous-GIS framing, including self-verification, used in 04_01.
* **Kao, C. H., et al. (2026).** [Towards LLM Agents for Earth Observation](https://aclanthology.org/2026.findings-acl.124/). *Findings of ACL 2026*, 2597-2611. **Peer-reviewed conference paper.** An EO-specific benchmark that helps distinguish execution success from scientific correctness.
* [Earth-Agent: official research implementation and paper links](https://github.com/opendatalab/Earth-Agent). **Companion research resource.** Useful for inspecting implementation; a repository is not itself independent validation.

Publication labels identify the cited version and, where stated, the peer-reviewed venue. An arXiv copy of an accepted paper is not a separate evaluation. Read methods, scope, and limitations before transferring findings to another application.
